In [16]:
import pandas as pd
df = pd.read_csv("../data/Delivery_Logistics_reconstructed.csv")

In [17]:
# Some timestamps use '.' instead of ':' for time (13.00 instead of 13:00)
# Pandas cannot parse this, so we fix the format first.

df["order_ts_recon"] = df["order_ts_recon"].str.replace(".", ":", regex=False)
df["expected_ts_recon"] = df["expected_ts_recon"].str.replace(".", ":", regex=False)

In [18]:
# Convert order timestamp
# dayfirst=True because format is DD-MM-YYYY
# errors="coerce" converts invalid values to NaT instead of crashing

df["order_ts_recon"] = pd.to_datetime(
    df["order_ts_recon"],
    dayfirst=True,
    errors="coerce"
)

# Convert expected delivery timestamp
df["expected_ts_recon"] = pd.to_datetime(
    df["expected_ts_recon"],
    dayfirst=True,
    errors="coerce"
)

In [19]:
# This column already follows ISO datetime format
# so no cleaning required

df["delivery_ts_recon"] = pd.to_datetime(
    df["delivery_ts_recon"],
    errors="coerce"
)

In [20]:
# Check datatypes after conversion
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 25 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   delivery_id                25000 non-null  float64       
 1   delivery_partner           25000 non-null  object        
 2   package_type               25000 non-null  object        
 3   vehicle_type               25000 non-null  object        
 4   delivery_mode              25000 non-null  object        
 5   region                     25000 non-null  object        
 6   weather_condition          25000 non-null  object        
 7   distance_km                25000 non-null  float64       
 8   package_weight_kg          25000 non-null  float64       
 9   delayed                    25000 non-null  object        
 10  delivery_status            25000 non-null  object        
 11  delivery_rating            25000 non-null  int64         
 12  deli

In [21]:
# -----------------------------
# Create target variables
# -----------------------------

# Regression target → how many hours early/late delivery happened
df["delay_hours"] = df["delay_hours_recon"]

# Classification target → delayed (1) or on-time (0)
df["delayed_flag"] = df["delayed_flag_recon"]

train_regression.py → uses delay_hours



train_classification.py → uses delayed_flag

In [22]:
# -----------------------------
# Drop leakage columns
# -----------------------------

leakage_columns = [
    "delivery_status",
    "delayed",
    "delay_hours_recon",
    "delayed_flag_recon",
    "delivery_ts_recon"
]

df = df.drop(columns=leakage_columns, errors="ignore")

In [23]:
# -----------------------------
# Drop identifier column
# -----------------------------

df = df.drop(columns=["delivery_id"], errors="ignore")

**Verifying After Cleaning**

In [24]:
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   delivery_partner           25000 non-null  object        
 1   package_type               25000 non-null  object        
 2   vehicle_type               25000 non-null  object        
 3   delivery_mode              25000 non-null  object        
 4   region                     25000 non-null  object        
 5   weather_condition          25000 non-null  object        
 6   distance_km                25000 non-null  float64       
 7   package_weight_kg          25000 non-null  float64       
 8   delivery_rating            25000 non-null  int64         
 9   delivery_cost              25000 non-null  float64       
 10  expected_time_hours_recon  25000 non-null  float64       
 11  speed_kmph_recon           25000 non-null  int64         
 12  weat

,delivery_partner,package_type,vehicle_type,delivery_mode,region,weather_condition,distance_km,package_weight_kg,delivery_rating,delivery_cost,...,speed_kmph_recon,weather_mult_recon,delivery_time_hours_recon,partner_mult_recon,order_date_recon,order_ts_recon,expected_ts_recon,hour,delay_hours,delayed_flag
0,amazon logistics,automobile parts,ev bike,standard,west,clear,235.6,48.07,5,1322.21,...,30,1.0,9.332489,1.001906,21-10-2024,2024-10-21 13:00:00,2024-10-23 17:48:00,13,-43.467511,0
1,amazon logistics,clothing,bike,express,central,stormy,81.8,45.51,2,595.53,...,35,1.1,4.129935,1.001906,02-01-2024,2024-01-02 12:00:00,2024-01-02 20:00:00,12,-3.870065,0
2,amazon logistics,clothing,van,same day,north,clear,282.9,31.33,2,1608.49,...,45,1.0,7.427398,1.001906,31-05-2024,2024-05-31 11:00:00,2024-06-01 13:24:00,11,-18.972602,0
3,amazon logistics,cosmetics,ev bike,two day,central,hot,88.6,8.67,3,469.01,...,30,1.1,3.997011,1.001906,03-01-2024,2024-01-03 17:00:00,2024-01-05 17:00:00,17,-44.002989,0
4,amazon logistics,cosmetics,ev van,two day,east,rainy,204.2,8.09,4,1045.27,...,40,1.2,6.933351,1.001906,19-03-2024,2024-03-19 13:00:00,2024-03-21 17:48:00,13,-45.866649,0


In [26]:
# --------------------------------------------------
# Create 'processed' folder automatically if missing
# --------------------------------------------------

import os

os.makedirs("../data/processed", exist_ok=True)

In [27]:
# --------------------------------------------------
# Save processed dataset as CSV
# --------------------------------------------------

df.to_csv(
    "../data/processed/processed_data.csv",
    index=False   # prevents saving row numbers
)

print("Processed dataset saved successfully!")

Processed dataset saved successfully!


**Interpretation**

Converted timestamps into proper datetime format

Created target variables (delay hours & delayed flag)

Removed columns that reveal future information

Extracted useful time features (hour, weekday, weekend)

Created new meaningful features (cost per km, duration etc.)

Removed unnecessary columns